# 6. From a question to a reproducible student project

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/06_student_project.ipynb)

A simulation project is more than a notebook that produces an
attractive image. It connects a biological question to a measurable
prediction, records the model and software version, tests invariants
and separates generated results from source files.

**Learning objectives**

- formulate a measurable simulation question;
- construct and save a complete ModelSpec;
- add a small registered interaction without editing BioLGCA internals;
- test the interaction on every model it supports; and
- organize files so another person can reproduce the run.


In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

from lgca import interaction
from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
    save_model_spec,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder
from lgca.testing import check_interaction

print("BioLGCA version:", version("biolgca"))


## Question, prediction and observable

**Question:** Does a deterministic clockwise channel rotation create
a reproducible circulation bias relative to unbiased random walk?

**Prediction:** repeated rotation changes the direction of flux but
does not change total particle number.

**Observables:** total population checks the conservation law, while
the summed flux vector measures directional change. The conservation
check should become an automated unit test when the interaction is
moved from exploration into a reusable project module.


## Define a small interaction

The rule below turns every moving cell clockwise by 90°. It receives
the lattice state, whose `counts` array holds the cells of every
channel with the shape `dims + (n_species, K)`, and replaces it by the
rotated state. It is a *reorientation*: it changes directions but keeps
the cells of every node, which the decorator checks after every step.
Because it only moves cell numbers between channels, it works with and
without volume exclusion and for any number of species.

This is project code, not a copy of a BioLGCA implementation. In a
real project, place it in `interactions.py`, import that module before
loading a model file, and test it (see below).


In [ ]:
@interaction(
    kind="reorientation",
    families=("classical", "nove"),
    geometries="square",
    name="tutorial.rotate_velocity_channels",
)
def rotate_velocity_channels(state):
    """Turn every moving cell clockwise by 90 degrees."""
    counts = state.counts.copy()
    # channels: east, north, west, south; a cell moving north now moves east
    counts[..., :4] = np.roll(counts[..., :4], shift=-1, axis=-1)
    state.counts = counts


# Running this cell again replaces the rule.
print(rotate_velocity_channels)


## Add the interaction to a visible specification

Registration gives the interaction a stable name. Calling the rule
gives the entry for the list of operators; a model file refers to the
same name, exactly as it would refer to a built-in interaction.
Propagation is disabled for the one-step invariant test so that the
interaction is isolated.


In [ ]:
project_spec = ModelSpec(
    description=Description(
        title="Clockwise channel rotation",
        details="Student project model used to test a conserving reorientation.",
        tags=("student-project", "custom-interaction"),
    ),
    space=SpaceSpec(
        geometry="square",
        dims=(10, 10),
        boundary="periodic",
    ),
    state=StateSpec(
        density=0.2,
        restchannels=0,
    ),
    time=TimeSpec(
        steps=1,
        seed=61,
    ),
    dynamics=InteractionPipelineSpec(
        operators=[rotate_velocity_channels()],
        propagation=False,
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

project_result = run_model(project_spec, showprogress=False)
before = project_result.data["nodes"][0]
after = project_result.data["nodes"][1]

assert before.sum() == after.sum()
expected = before.copy()
expected[..., :4] = before[..., [1, 2, 3, 0]]
np.testing.assert_array_equal(after, expected)
np.testing.assert_array_equal(after[..., 4:], before[..., 4:])
assert project_result.data["population"][0] == project_result.data["population"][-1]
print("conserved population:", project_result.data["population"].tolist())


In [ ]:
before_flux = project_result.lgca.calc_flux(before).sum(axis=(0, 1))
after_flux = project_result.lgca.calc_flux(after).sum(axis=(0, 1))
np.testing.assert_allclose(after_flux, [before_flux[1], -before_flux[0]])
print("flux before:", before_flux)
print("flux after: ", after_flux)

fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)
axes[0].imshow(before.sum(axis=-1).T, origin="lower")
axes[0].set_title("before interaction")
axes[1].imshow(after.sum(axis=-1).T, origin="lower")
axes[1].set_title("after interaction")
plt.show()
plt.close(fig)


The density maps are identical because channel rotation changes
direction, not node occupancy. The flux vectors reveal the changed
channel state. This illustrates why an invariant and a question-
specific observable test different aspects of an interaction.


## Test the interaction on every model it supports

The assertions above check one model. `check_interaction` runs the rule
on small random models of every lattice and family it declares, with
one and two species and with periodic and reflecting boundaries, and
checks that states stay valid, that the conservation law of its kind
holds and that the same seed gives the same result. It raises an error
with a readable report when something is wrong, so in a project it
becomes a one-line test in `tests/test_interactions.py`.


In [ ]:
report = check_interaction(rotate_velocity_channels)
print(report)


## Save the model and provenance

A shareable run records the model specification, random seed and
package version. A custom plugin's Python module must accompany the
model file; importing trusted plugin code is deliberately separate
from parsing model data.


In [ ]:
with TemporaryDirectory() as temporary_directory:
    run_directory = Path(temporary_directory)
    model_path = save_model_spec(project_spec, run_directory / "model.json")
    metadata = {
        "biolgca_version": version("biolgca"),
        "seed": project_spec.time.seed,
        "model_file": model_path.name,
        "interaction_module": "interactions.py",
    }
    print(model_path.read_text(encoding="utf-8")[:400] + "...")
    print(metadata)


## Suggested project layout

```text
my-lgca-project/
|-- README.md                 # question, prediction and run commands
|-- model.json                # saved ModelSpec
|-- interactions.py           # registered custom interaction
|-- analysis.py               # reusable observables
|-- notebooks/
|   `-- exploration.ipynb     # exploratory narrative, no hidden state
|-- tests/
|   `-- test_interactions.py  # conservation and compatibility tests
`-- results/                  # generated files, normally ignored by git
```

Record the BioLGCA version and seeds with every result. Keep raw
simulation output separate from analysis figures so results can be
regenerated without editing source files.

## Project checklist

1. State a biological question and measurable prediction.
2. Start by composing built-in terms and phases visibly.
3. Add custom code only when the mechanism is genuinely missing.
4. Test conservation laws and supported model families.
5. Choose observables before inspecting the most favorable run.
6. Use multiple prespecified seeds and report variability.
7. Save ModelSpec, package version, custom plugin module and analysis code.

## Exercises

1. Re-enable propagation and predict the combined effect of rotation
   and movement.
2. Add a parameter `clockwise=True` that chooses the direction of the
   rotation, and check both directions with `check_interaction`.
3. Move the rule into a module and write a pytest file whose test calls
   `check_interaction`.
4. Write a movement bias instead of a deterministic rule: a function
   decorated with `@reorientation_term(coupling="flux")` that returns,
   at every node, the direction perpendicular to the line from the
   lattice centre, so cells circle around the centre. Combine it with
   `random_walk` in a `ReorientationSpec` and measure the circulation.
